In [1]:
import pandas as pd
import numpy as np

In [13]:
df = pd.read_csv('../../data/raw/meteo_departements_senegal_2021_2026.csv')
df.head()

,DRS,DISTRICT,PERIODE,temperature_moyenne,temperature_max,temperature_min,precipitation,humidite,vent,rayonnement_solaire
0,DRS Dakar,Dakar Centre,Janvier 2021,21.04,23.28,19.53,0.06,71.73,5.40,5.13
1,DRS Dakar,Dakar Centre,Février 2021,19.87,22.08,18.72,1.68,78.41,5.11,5.25
2,DRS Dakar,Dakar Centre,Mars 2021,19.75,21.10,18.93,8.03,84.41,5.43,6.81
3,DRS Dakar,Dakar Centre,Avril 2021,20.20,21.21,19.62,0.39,89.08,4.79,7.11
4,DRS Dakar,Dakar Centre,Mai 2021,21.00,21.63,20.55,0.97,89.98,4.55,6.59


In [3]:
# Convert French month names to English
month_map = {
    "Janvier": "January",
    "Février": "February",
    "Mars": "March",
    "Avril": "April",
    "Mai": "May",
    "Juin": "June",
    "Juillet": "July",
    "Août": "August",
    "Septembre": "September",
    "Octobre": "October",
    "Novembre": "November",
    "Décembre": "December"
}

# Replace French month names
for fr, en in month_map.items():
    df["PERIODE"] = df["PERIODE"].str.replace(fr, en, regex=False)

# Convert to datetime
df["PERIODE"] = pd.to_datetime(df["PERIODE"], format="%B %Y")

# Split the data from 2021 to 2024 for training and 2025 for test to 2026 for remaining data
train_df = df[df["PERIODE"].dt.year.isin([2021, 2022, 2023, 2024])]
test_df = df[df["PERIODE"].dt.year.isin([2025])]
test_remain = df[df["PERIODE"].dt.year.isin([2026])]

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Remaining test shape:", test_remain.shape)

Training shape: (3792, 10)
Test shape: (948, 10)
Remaining test shape: (632, 10)


In [4]:
# Save if needed
train_df.to_csv("../../data/raw/meteo_departements_senegal_2021_2024_train.csv", index=False)
test_df.to_csv("../../data/raw/meteo_departements_senegal_2025_test.csv", index=False)
test_remain.to_csv("../../data/raw/meteo_departements_senegal_2026_test_remain.csv", index=False)

Add weather value to processed

In [ ]:
# Read data in ../../data/processed/Base_PEC_MALARIA_CS_2021_2024.xlsx
df_processed = pd.read_excel('../../data/raw/Base_PEC_MALARIA_CS_2021_2024.xlsx')

# Replace French month names
for fr, en in month_map.items():
    df_processed["PERIODE"] = df_processed["PERIODE"].str.replace(fr, en, regex=False)

# Convert to datetime
df_processed["PERIODE"] = pd.to_datetime(df_processed["PERIODE"], format="%B %Y")


df_processed.head()

,DRS,DISTRICT,PERIODE,Nombre de sites PECADOM fonctionnels,Palu:ACT NRSS-Traitement Dispensé,Cas Suspect Palu hospitalisations,Nombre de cas de paludisme GRAVE confirmes et traites qui ont beneficie dune GE de controle,Femmes enceintes recu TPI1/SP1,Traitements ACT dispensés à Nourrisson (2 à 11 mois),R20_DSME_Premier contact,...,Palu:Artésunate injectable-Stock reçu,Palu:ACT grand enfant-Nombre jours de rupture consecutif,Palu:Quinine 400-Traitement Dispensé,CAS correctement pris en charge conformément aux directives niveau case,Palu:Artésunate injectable-Stock début,Cas palu référés dont TDR Négatifs,Lames Positifs (GE) palu,Palu:ACT adulte-Traitement Dispensé,DSDOM ayant notifié,date_downloaded
0,DRS Tambacounda,Bakel,2025-08-01,11.0,NaN,NaN,NaN,NaN,NaN,45.0,...,NaN,NaN,NaN,4.0,NaN,7.0,NaN,57.0,11.0,2025-10-08
1,DRS Diourbel,Bambey,2025-08-01,10.0,1.0,33.0,NaN,NaN,NaN,506.0,...,NaN,NaN,NaN,NaN,118.0,10.0,7.0,55.0,9.0,2025-10-08
2,DRS Ziguinchor,Bignona,2025-08-01,NaN,NaN,33.0,NaN,NaN,NaN,188.0,...,NaN,NaN,NaN,NaN,145.0,NaN,NaN,7.0,NaN,2025-10-08
3,DRS Kaffrine,Birkelane,2025-08-01,15.0,NaN,NaN,NaN,NaN,NaN,218.0,...,NaN,NaN,NaN,NaN,NaN,43.0,4.0,22.0,11.0,2025-10-08
4,DRS Sedhiou,Bounkiling,2025-08-01,48.0,NaN,NaN,NaN,NaN,NaN,31.0,...,NaN,NaN,NaN,NaN,NaN,12.0,NaN,11.0,15.0,2025-10-08


In [10]:
test_df.head()

,DRS,DISTRICT,PERIODE,temperature_moyenne,temperature_max,temperature_min,precipitation,humidite,vent,rayonnement_solaire
48,DRS Dakar,Dakar Centre,2025-01-01,22.05,24.59,20.16,0.03,68.95,5.28,5.51
49,DRS Dakar,Dakar Centre,2025-02-01,20.03,22.38,18.42,0.29,73.98,5.45,6.06
50,DRS Dakar,Dakar Centre,2025-03-01,19.53,20.21,19.04,52.58,86.47,6.02,6.42
51,DRS Dakar,Dakar Centre,2025-04-01,20.56,21.49,20.00,0.59,90.18,5.07,6.74
52,DRS Dakar,Dakar Centre,2025-05-01,22.24,22.84,21.79,4.74,90.28,4.39,6.41


In [ ]:
# Insert train_df to df_processed before R20_DSME_Femmes enceintes ayant reçu TPI 2 column where DRS, PERIODE and DEPARTEMENT match
# Merge on the matching keys
merged_df = df_processed.merge(
    train_df,
    on=["DRS", "PERIODE", "DISTRICT"],
    how="left"
)

# Columns to insert (exclude the key columns)
new_cols = [
    col for col in train_df.columns
    if col not in ["DRS", "PERIODE", "DISTRICT"]
]

# Find insertion position
insert_pos = merged_df.columns.get_loc("R20_DSME_Femmes enceintes ayant reçu TPI 2")

# Reorder columns
cols = (
    list(merged_df.columns[:insert_pos])
    + new_cols
    + [c for c in merged_df.columns[insert_pos:] if c not in new_cols]
)

merged_df = merged_df[cols]

In [ ]:
# Save the merged DataFrame to a new Excel file
merged_df.to_excel('../../data/features/Base_MALARIA_CS_CLEAN_2021_2024_with_weather.xlsx', index=False)